# ⚙️ Clase 4 — Feature Engineering (Ingeniería de Características)
### Encoding · Variables Sintéticas · Selección con SelectKBest

---

## ¿Qué es Feature Engineering y por qué es crítico?

El **Feature Engineering** es el proceso de transformar los datos crudos en variables que un modelo de machine learning pueda entender y aprovechar al máximo.

> *"Los algoritmos de ML aprenden de los datos que les das. Si les das datos mal preparados, aprenden mal. Si les das buenas variables, aprenden bien."*

Es considerado por muchos practicantes el paso que más impacto tiene en la calidad de un modelo — más que el algoritmo elegido.

---

## ¿Qué haremos hoy?

```
Datos crudos (texto, categorías, números sin transformar)
        │
        ▼
┌─────────────────────────────────────────────┐
│  1. ENCODING          → texto a números     │
│  2. FEATURE CREATION  → variables nuevas    │
│  3. FEATURE SELECTION → quedarse con las    │
│                         que más aportan     │
└─────────────────────────────────────────────┘
        │
        ▼
Dataset listo para modelar
```

---

## Dataset — Sample Superstore

El mismo dataset de clases anteriores. Hoy lo usamos como punto de partida para construir un dataset enriquecido y optimizado para modelado.

**Variables categóricas que vamos a codificar:**

| Variable | Valores únicos | Tipo de encoding apropiado |
|---|---|---|
| `Ship Mode` | 4 (tiene orden implícito) | Ordinal |
| `Segment` | 3 (sin orden) | One-Hot |
| `Region` | 4 (sin orden) | One-Hot |
| `Category` | 3 (sin orden) | One-Hot |
| `Sub-Category` | 17 (sin orden, alta cardinalidad) | One-Hot con precaución |

> ⏱️ Duración estimada: **~2 horas**

---
## 📦 Sección 1 — Importaciones

In [ ]:
import pandas as pd                          # manipulación de DataFrames
import numpy as np                           # operaciones matemáticas sobre arrays
import matplotlib.pyplot as plt              # motor base de visualización
import seaborn as sns                        # gráficas estadísticas de alto nivel

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder  # encoders de variables categóricas
from sklearn.feature_selection import SelectKBest, f_regression, f_classif  # selección de features

---
## 📂 Sección 2 — Carga del dataset

In [ ]:
# Lee el archivo CSV con encoding latin1 para evitar errores con caracteres especiales
df = pd.read_csv('SampleSuperstore.csv', encoding='latin1')

# Conserva solo las columnas útiles; descarta City, State, Country (alta cardinalidad sin valor directo)
# y Postal Code (es un ID geográfico, no una variable numérica real)
cols_utiles = ['Ship Mode', 'Segment', 'Region', 'Category',
               'Sub-Category', 'Sales', 'Quantity', 'Discount', 'Profit']

# .copy() crea una copia independiente del DataFrame; evita el SettingWithCopyWarning al modificarlo
df = df[cols_utiles].copy()

# Vista rápida del dataset cargado
df.head(4)

---
## 🔤 Sección 3 — Encoding de variables categóricas

### ¿Por qué los modelos de ML no aceptan texto directamente?

Los algoritmos de machine learning operan con **operaciones matemáticas**: sumas, multiplicaciones, distancias, derivadas.
Estas operaciones no tienen sentido con texto. No puedes calcular *`'Consumer' × 0.3`*.

El encoding convierte el texto en números de forma que el modelo pueda procesarlo — pero **la forma en que conviertes importa muchísimo**.

---

### Los tres métodos principales de encoding

| Método | ¿Qué produce? | Riesgo | Cuándo usarlo |
|---|---|---|---|
| **Label Encoding** | Un número por categoría (0, 1, 2...) | El modelo asume orden artificial (2 > 1 > 0) | Solo si la variable **tiene orden real** |
| **One-Hot Encoding** | Una columna binaria por categoría | Muchas columnas con alta cardinalidad | Variables nominales (sin orden) con pocas categorías |
| **Ordinal Encoding** | Un número por categoría respetando el orden real | Ninguno si el orden está bien definido | Variables con orden explícito (Bajo < Medio < Alto) |

> ⚠️ **El error más común:** usar Label Encoding en variables nominales como `Region` o `Category`.
> Si codificas `East=0`, `West=1`, `Central=2`, `South=3`, el modelo pensará que `South` vale el triple que `East` — lo cual es completamente falso.

### 3.1 — Label Encoding

`LabelEncoder` asigna un entero a cada categoría en **orden alfabético**.
Es simple y produce una sola columna, pero introduce un **orden falso** en variables nominales.

**¿Cuándo es correcto usarlo?**
Solo cuando la variable tiene un orden real: calificación (Malo < Regular < Bueno) o prioridad (Baja < Media < Alta).
Lo usamos aquí solo para demostrar cómo funciona y entender el problema que genera.

In [ ]:
# Instancia el codificador; aprende las categorías y les asigna enteros alfabéticamente
le = LabelEncoder()

# fit_transform: aprende las categorías (fit) y las convierte a enteros (transform) en un paso
# Resultado: 'Central'→0, 'East'→1, 'South'→2, 'West'→3
df['Region_label'] = le.fit_transform(df['Region'])

# Construye el diccionario de mapeo para ver qué entero corresponde a qué categoría
mapeo_label = dict(zip(le.classes_, le.transform(le.classes_)))

# Muestra el mapeo y las primeras filas para verificar el resultado
print('Mapeo Label Encoding — Region:', mapeo_label)
print()
print('⚠️  Problema: el modelo interpretará West(3) > South(2) > East(1) > Central(0)')
print('   Eso es matemáticamente falso para una variable geográfica sin orden real.')

### 3.2 — Ordinal Encoding

`OrdinalEncoder` también produce un número por categoría, pero permite especificar **el orden correcto tú mismo**.
Esto es diferente al Label Encoding: tú controlas qué número corresponde a qué categoría.

**¿Cuándo usarlo?**
Cuando la variable tiene un orden real y conocido. En este dataset, `Ship Mode` tiene un orden implícito de velocidad:
`Standard Class (0) < Second Class (1) < First Class (2) < Same Day (3)`

In [ ]:
# Define el orden explícito de las categorías de menor a mayor prioridad/velocidad
# El índice de cada elemento en la lista determina el entero asignado (0, 1, 2, 3)
orden_envio = [['Standard Class', 'Second Class', 'First Class', 'Same Day']]

# OrdinalEncoder respeta el orden definido por el usuario, no el alfabético
oe = OrdinalEncoder(categories=orden_envio)

# fit_transform recibe un array 2D (doble corchete) → devuelve un array 2D con los enteros
# reshape(-1) convierte el array 2D de una sola columna a un vector 1D para asignarlo al DataFrame
df['Ship_Mode_ordinal'] = oe.fit_transform(df[['Ship Mode']]).reshape(-1)

# Verifica el mapeo: Standard Class debe ser 0 y Same Day debe ser 3
df[['Ship Mode', 'Ship_Mode_ordinal']].drop_duplicates().sort_values('Ship_Mode_ordinal')

### 3.3 — One-Hot Encoding con `pd.get_dummies`

One-Hot Encoding crea **una columna binaria por cada categoría**. Si un registro pertenece a esa categoría, la columna vale 1; si no, vale 0.

```
Region      →    Region_East  Region_West  Region_Central  Region_South
'East'      →         1           0             0              0
'West'      →         0           1             0              0
```

**¿Por qué `drop_first=True`?**
Si tienes 4 regiones y ya conoces los valores de 3 columnas, la cuarta siempre se puede deducir.
Esa redundancia se llama **multicolinealidad perfecta** y puede causar problemas en algunos modelos.
Con `drop_first=True` eliminamos una columna por variable; esa categoría eliminada queda como **referencia implícita**.

**`pd.get_dummies` vs `sklearn.OneHotEncoder`:**
- `pd.get_dummies` → más simple, integrado en pandas, ideal para exploración y notebooks
- `sklearn.OneHotEncoder` → necesario para pipelines de ML porque guarda las categorías aprendidas y puede aplicarlas a datos nuevos sin reentrenar

In [ ]:
# get_dummies aplica OHE a las columnas indicadas en 'columns'
# drop_first=True elimina la primera categoría de cada variable (la de referencia) para evitar redundancia
# dtype=int convierte los valores True/False resultantes a 0/1 (más legible)
df_ohe = pd.get_dummies(
    df,
    columns=['Segment', 'Region', 'Category'],
    drop_first=True,
    dtype=int
)

# Filtra los nombres de las columnas nuevas generadas por OHE
cols_nuevas = [c for c in df_ohe.columns if c.startswith(('Segment_', 'Region_', 'Category_'))]

# Muestra solo las columnas OHE para verificar la codificación
df_ohe[cols_nuevas].head(4)

### 3.4 — Visualización: Label Encoding vs One-Hot Encoding

Esta gráfica muestra el problema concreto del Label Encoding en variables nominales y cómo OHE lo resuelve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Izquierda: Label Encoding — muestra el orden artificial asignado ───────
orden = df[['Region','Region_label']].drop_duplicates().sort_values('Region_label')  # ordena por el entero asignado
axes[0].barh(orden['Region'], orden['Region_label'], color='tomato', edgecolor='white')  # barras horizontales
axes[0].set_title('Label Encoding — asigna un orden falso a Region')
axes[0].set_xlabel('Valor numérico asignado')
axes[0].axvline(0, color='black', linewidth=0.5)  # línea de referencia en 0

# ── Derecha: One-Hot — muestra la representación binaria correcta ──────────
cols_region = [c for c in df_ohe.columns if c.startswith('Region_')]  # extrae solo las columnas OHE de Region
muestra_ohe = (df_ohe[['Region'] + cols_region]   # selecciona Region + sus columnas OHE
               .drop_duplicates()                  # una fila por categoría única
               .sort_values('Region')              # ordena alfabéticamente
               .set_index('Region'))               # pone Region como índice para el heatmap

sns.heatmap(muestra_ohe, annot=True, fmt='d',     # fmt='d' → enteros, no decimales
            cmap='Blues', cbar=False,              # sin barra de color lateral (no aporta aquí)
            linewidths=0.5, ax=axes[1])
axes[1].set_title('One-Hot Encoding — representación binaria sin orden artificial')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

---
## 🧪 Sección 4 — Creación de variables sintéticas

### ¿Qué es una variable sintética?

Una variable sintética (también llamada *engineered feature*) es una **columna nueva que creas a partir de las existentes**, combinándolas matemáticamente o transformándolas.

**¿Por qué crear variables que ya están "implícitas" en los datos?**
Los modelos de ML aprenden patrones en el espacio de features que les das. Si el patrón importante es un ratio o una interacción, y no se lo das explícitamente, el modelo tiene que descubrirlo solo — lo cual es más difícil y requiere más datos.

Tú, con conocimiento del negocio, puedes codificar esa lógica directamente.

---

### Tipos de variables sintéticas

| Tipo | Ejemplo | Cuándo crearla |
|---|---|---|
| **Ratio / proporción** | `Profit / Sales` | Cuando importa la relación relativa, no el valor absoluto |
| **Producto / interacción** | `Sales × Quantity` | Cuando el efecto combinado importa más que cada parte |
| **Binaria (flag)** | `1 si Profit > 0` | Para convertir un problema de regresión en clasificación |
| **Agrupada (bin)** | `rango de descuento` | Para capturar efectos no lineales con tramos definidos |
| **Diferencia** | `Sales - Profit` | Para capturar el costo implícito de una transacción |

### 4.1 — Variables de ratio

Los ratios normalizan una variable respecto a otra, capturando la **eficiencia relativa** en lugar del valor bruto.

`profit_margin = Profit / Sales` → el margen de ganancia sobre cada venta.
Una venta de \$10 000 con \$100 de profit tiene margen del 1 %. Una venta de \$50 con \$20 tiene margen del 40 %.
El valor bruto de `Sales` oculta esa diferencia; el ratio la revela.

In [ ]:
# Copia limpia del dataset original para construir el dataset de features
df_feat = df[['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category',
              'Sales', 'Quantity', 'Discount', 'Profit']].copy()

# Margen de ganancia: proporción del profit sobre las ventas brutas
# np.where evita la división por cero: si Sales == 0 asigna 0 directamente
df_feat['profit_margin'] = np.where(
    df_feat['Sales'] != 0,          # condición: Sales debe ser distinto de 0
    df_feat['Profit'] / df_feat['Sales'],  # cálculo normal cuando Sales > 0
    0                                       # valor alternativo para evitar ZeroDivisionError
)

# Ingresos totales: precio unitario × unidades vendidas
df_feat['revenue_total'] = df_feat['Sales'] * df_feat['Quantity']

# Costo implícito de la transacción: lo que costó vender ese pedido
df_feat['costo_implicito'] = df_feat['Sales'] - df_feat['Profit']

# Vista rápida de las tres nuevas columnas junto a sus fuentes
df_feat[['Sales', 'Profit', 'Quantity', 'profit_margin', 'revenue_total', 'costo_implicito']].head(5)

### 4.2 — Variables binarias (flags)

Una variable binaria codifica si una condición se cumple con **1 (sí) o 0 (no)**.
Son útiles tanto para enriquecer el dataset como para reformular problemas.

Con `es_rentable` convertimos la pregunta:
*"¿Cuánto profit genera esta transacción?"* (regresión) → *"¿Esta transacción genera ganancia?"* (clasificación binaria)

In [ ]:
# Variable target de clasificación: 1 si la transacción fue rentable, 0 si tuvo pérdida
# (Profit > 0) devuelve True/False → .astype(int) lo convierte a 1/0
df_feat['es_rentable'] = (df_feat['Profit'] > 0).astype(int)

# Flag: indica si se aplicó algún descuento en la transacción
df_feat['tiene_descuento'] = (df_feat['Discount'] > 0).astype(int)

# Flag: pedido de alto valor — por encima del percentil 75 de Sales
umbral_p75 = df_feat['Sales'].quantile(0.75)            # calcula el p75 dinámicamente sobre los datos reales
df_feat['alto_valor'] = (df_feat['Sales'] > umbral_p75).astype(int)  # 1 si supera el umbral, 0 si no

# Resumen de las tres variables binarias creadas
resumen_flags = pd.DataFrame({
    'flag':        ['es_rentable', 'tiene_descuento', 'alto_valor'],
    'casos_con_1': [df_feat['es_rentable'].sum(),
                    df_feat['tiene_descuento'].sum(),
                    df_feat['alto_valor'].sum()],
    'porcentaje':  [f"{df_feat['es_rentable'].mean():.1%}",
                    f"{df_feat['tiene_descuento'].mean():.1%}",
                    f"{df_feat['alto_valor'].mean():.1%}"]
})
resumen_flags

### 4.3 — Variables de agrupación (binning)

El **binning** convierte una variable numérica continua en rangos discretos.
Útil cuando la relación con el target no es lineal: no importa si el descuento es 21 % o 23 %, lo que importa es que está en el rango medio.

- `pd.cut` → divide por **rangos de valor** definidos manualmente
- `pd.qcut` → divide por **cuantiles** (igual número de observaciones en cada grupo)

In [ ]:
# pd.cut divide el rango de Discount en intervalos definidos por el analista
# bins: bordes de cada intervalo; labels: nombre legible de cada tramo
df_feat['rango_descuento'] = pd.cut(
    df_feat['Discount'],
    bins=[-0.01, 0.0, 0.2, 0.4, 0.8],               # -0.01 para incluir el 0 en el primer tramo
    labels=['Sin descuento', 'Bajo', 'Medio', 'Alto']  # etiquetas en el mismo orden que los tramos
)

# pd.qcut divide en cuartiles: Q1 tiene el 25% más bajo, Q4 el 25% más alto
df_feat['cuartil_ventas'] = pd.qcut(
    df_feat['Sales'],
    q=4,                                              # 4 grupos de igual tamaño poblacional
    labels=['Q1 (bajo)', 'Q2', 'Q3', 'Q4 (alto)']
)

# Distribución de transacciones por rango de descuento para verificar el binning
df_feat['rango_descuento'].value_counts().sort_index()

### 4.4 — Validación visual de las variables sintéticas

Antes de usar una variable sintética en un modelo, siempre hay que validar que **aporta información real**.
Si la distribución es idéntica entre grupos, la variable no está añadiendo valor.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Izquierda: profit_margin por Category ──────────────────────────────────
# Si el margen varía entre categorías, confirma que la variable sintética captura algo útil
sns.boxplot(data=df_feat, x='Category', y='profit_margin',
            palette='Set2', width=0.5, ax=axes[0])
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)   # línea de breakeven en 0
axes[0].set_title('Margen de ganancia por Categoría
(variable sintética: Profit / Sales)')
axes[0].set_xlabel('')
axes[0].set_ylabel('Profit Margin (ratio)')

# ── Derecha: profit promedio por rango de descuento ────────────────────────
# Si el profit cae de forma consistente al aumentar el rango, el binning captura el patrón
profit_rango = df_feat.groupby('rango_descuento', observed=True)['Profit'].mean()  # observed=True evita warning con Categorical
profit_rango.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white', rot=0)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Profit promedio por rango de descuento
(variable sintética: rango_descuento)')
axes[1].set_xlabel('')
axes[1].set_ylabel('Profit promedio ($)')

plt.tight_layout()
plt.show()

---
## 🎯 Sección 5 — Selección de características: SelectKBest

### ¿Por qué seleccionar features en lugar de usar todas?

Más variables no siempre es mejor. Agregar variables poco informativas o redundantes puede causar:

- **Sobreajuste (overfitting):** el modelo aprende ruido que no existe en datos nuevos
- **Maldición de la dimensionalidad:** con muchas features los datos se vuelven muy dispersos
- **Mayor tiempo de entrenamiento y modelos menos interpretables**

La selección de features mantiene solo las **que más aportan** para predecir el target.

---

### Métodos de selección de características

```
Métodos de selección
├── Filter methods    → evalúan cada feature de forma independiente (estadística)
│   └── SelectKBest, correlación con target
├── Wrapper methods   → prueban subconjuntos de features entrenando modelos
│   └── RFE (Recursive Feature Elimination)
└── Embedded methods  → la selección ocurre durante el entrenamiento
    └── Lasso, feature_importances_ de árboles
```

Hoy trabajamos con **SelectKBest** (Filter method): el más rápido y sencillo de entender.

---

### ¿Cómo funciona SelectKBest?

1. Para cada feature, calcula un **score estadístico** que mide qué tan relacionada está con el target
2. Ordena las features de mayor a menor score
3. Selecciona las **K mejores**

La función estadística depende del tipo de target:
- `f_regression` → para targets numéricos continuos (predecir el Profit en $)
- `f_classif`   → para targets de clasificación (predecir si es rentable o no)

### 5.1 — Preparar el dataset numérico para SelectKBest

SelectKBest requiere que **todas las columnas sean numéricas**. Aplicamos OHE a las categóricas e incluimos las variables sintéticas creadas.

In [ ]:
# Aplica OHE a todas las variables categóricas del dataset enriquecido
# Se excluyen rango_descuento y cuartil_ventas porque son Categorical de pandas, no int
df_modelo = pd.get_dummies(
    df_feat.drop(columns=['rango_descuento', 'cuartil_ventas']),  # descarta los bins categóricos
    columns=['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category'],  # columnas a codificar
    drop_first=True,    # elimina la categoría de referencia para evitar multicolinealidad
    dtype=int           # convierte True/False a 0/1
)

# Define los dos targets (uno para regresión, uno para clasificación)
target_reg = 'Profit'       # target continuo: valor en dólares
target_clf = 'es_rentable'  # target binario: 0 o 1

# Features: todas las columnas excepto los dos targets
features_cols = [c for c in df_modelo.columns if c not in [target_reg, target_clf]]

X     = df_modelo[features_cols]   # matriz de features: shape (9994, n_features)
y_reg = df_modelo[target_reg]      # vector target de regresión
y_clf = df_modelo[target_clf]      # vector target de clasificación

# Muestra el shape final del dataset listo para SelectKBest
X.shape

### 5.2 — SelectKBest con `f_regression` (target numérico)

`f_regression` calcula la **correlación lineal** entre cada feature y el target numérico.
Devuelve un F-score (proporcional al cuadrado de la correlación de Pearson) y un p-valor.
Un F-score alto indica que esa feature tiene una relación lineal fuerte con el target.

In [ ]:
# k='all' calcula el score de TODAS las features para ver el ranking completo antes de filtrar
selector_reg = SelectKBest(score_func=f_regression, k='all')

# fit calcula los F-scores y p-valores para cada feature comparada contra y_reg (Profit)
selector_reg.fit(X, y_reg)

# Construye el DataFrame de resultados con nombre, score y p-valor de cada feature
scores_reg = pd.DataFrame({
    'feature': X.columns,              # nombre de cada columna
    'f_score': selector_reg.scores_,   # F-score: mayor = más relacionada con el target
    'p_valor': selector_reg.pvalues_   # p-valor: < 0.05 indica significancia estadística
}).sort_values('f_score', ascending=False)   # ordena de mayor a menor relevancia

# Muestra el top 15 para no saturar la salida
scores_reg.head(15).round(3)

In [ ]:
# Visualiza el ranking de las 15 features más relevantes para predecir Profit
top15_reg = scores_reg.head(15)

fig, ax = plt.subplots(figsize=(8, 5))

# Colorea en verde las significativas (p<0.05) y en gris las que no lo son
colores = ['#4dac26' if p < 0.05 else '#aaaaaa' for p in top15_reg['p_valor']]

# [::-1] invierte el orden para que la más importante quede arriba en el gráfico horizontal
ax.barh(top15_reg['feature'][::-1], top15_reg['f_score'][::-1],
        color=colores[::-1], edgecolor='white')

ax.set_title('Top 15 features — F-score para predecir Profit ($)
verde = estadísticamente significativa (p < 0.05)')
ax.set_xlabel('F-score (mayor = más relevante)')
plt.tight_layout()
plt.show()

### 5.3 — SelectKBest con `f_classif` (target de clasificación)

`f_classif` usa el **ANOVA F-test**: compara si la media de cada feature es significativamente diferente entre las clases (rentable vs no rentable).
Si la media de `Discount` es muy distinta entre pedidos rentables y no rentables, esa feature tiene un F-score alto.

In [ ]:
# Mismo proceso que f_regression pero orientado a un target binario (0/1)
selector_clf = SelectKBest(score_func=f_classif, k='all')

# fit calcula los F-scores comparando cada feature entre las dos clases de y_clf
selector_clf.fit(X, y_clf)

# Construye el DataFrame de resultados para el target de clasificación
scores_clf = pd.DataFrame({
    'feature': X.columns,
    'f_score': selector_clf.scores_,
    'p_valor': selector_clf.pvalues_
}).sort_values('f_score', ascending=False)

scores_clf.head(15).round(3)

In [ ]:
top15_clf = scores_clf.head(15)

fig, ax = plt.subplots(figsize=(8, 5))

# Colorea en azul las significativas y en gris las que no alcanzan significancia estadística
colores_clf = ['#2166ac' if p < 0.05 else '#aaaaaa' for p in top15_clf['p_valor']]

ax.barh(top15_clf['feature'][::-1], top15_clf['f_score'][::-1],
        color=colores_clf[::-1], edgecolor='white')

ax.set_title('Top 15 features — F-score para clasificar Rentable vs No Rentable
azul = estadísticamente significativa (p < 0.05)')
ax.set_xlabel('F-score (mayor = más discriminante entre clases)')
plt.tight_layout()
plt.show()

### 5.4 — Comparación: ¿las mismas features importan para ambos targets?

Es común que las features más relevantes para predecir el **valor exacto** de una variable no sean las mismas que para predecir si esa variable es **positiva o negativa**.
Esta comparación permite elegir features robustas para ambos enfoques.

In [ ]:
# Une los dos rankings por nombre de feature para comparar los F-scores lado a lado
comparacion = (
    scores_reg[['feature','f_score']]
    .rename(columns={'f_score': 'f_regresion'})        # renombra para distinguir las dos columnas
    .merge(
        scores_clf[['feature','f_score']].rename(columns={'f_score': 'f_clasificacion'}),
        on='feature'
    )
    .sort_values('f_regresion', ascending=False)        # ordena por relevancia en regresión
    .head(12)                                           # top 12 para que la gráfica sea legible
)

fig, ax = plt.subplots(figsize=(9, 5))

x     = np.arange(len(comparacion))   # posiciones numéricas en el eje X
ancho = 0.38                          # ancho de cada barra (las dos juntas ocupan ~0.76 del espacio)

# Barra izquierda: F-score de regresión
ax.bar(x - ancho/2, comparacion['f_regresion'],
       width=ancho, label='Regresión (predecir Profit $)', color='steelblue', edgecolor='white')

# Barra derecha: F-score de clasificación
ax.bar(x + ancho/2, comparacion['f_clasificacion'],
       width=ancho, label='Clasificación (predecir es_rentable)', color='seagreen', edgecolor='white')

ax.set_xticks(x)                                                          # posiciona los ticks
ax.set_xticklabels(comparacion['feature'], rotation=35, ha='right', fontsize=9)  # etiquetas rotadas
ax.set_title('F-score comparado — Regresión vs Clasificación · Top 12 features')
ax.set_ylabel('F-score')
ax.legend()
plt.tight_layout()
plt.show()

### 5.5 — Dataset final: aplicar SelectKBest y exportar

Una vez identificadas las mejores features, `SelectKBest` filtra el dataset automáticamente seleccionando solo las **K columnas con mayor F-score**.
Este dataset transformado es el que pasarías directamente a un algoritmo de ML.

In [ ]:
K = 10   # número de features a conservar — hiperparámetro ajustable según el problema

# Ajusta el selector para mantener exactamente K features con mayor F-score
selector_final = SelectKBest(score_func=f_regression, k=K)
selector_final.fit(X, y_reg)          # calcula los scores y determina cuáles K conservar

# get_support() devuelve una máscara booleana: True en las K posiciones seleccionadas
mascara = selector_final.get_support()

# Aplica la máscara al listado de nombres de columnas para obtener los nombres de las K seleccionadas
features_seleccionadas = X.columns[mascara].tolist()

# Dataset final: solo las K columnas seleccionadas
X_final = X[features_seleccionadas]   # filtra el DataFrame de features
y_final = y_reg                       # el target no cambia

# Vista del resultado final
print(f'Features seleccionadas (top {K}):')
for i, f in enumerate(features_seleccionadas, 1):
    print(f'  {i:2}. {f}')

In [ ]:
# Vista final del dataset listo para modelar
X_final.head(4)

---
## 🏁 Resumen de la clase

### Pipeline de Feature Engineering que construiste hoy

```
Dataset crudo
    │
    ├─ Encoding ──────────────────────────────────────────────────────────────
    │    ├── OrdinalEncoder  → Ship Mode (tiene orden real: velocidad de envío)
    │    ├── LabelEncoder    → demostración del error en variables nominales
    │    └── pd.get_dummies  → Segment, Region, Category, Sub-Category
    │
    ├─ Feature Creation ──────────────────────────────────────────────────────
    │    ├── Ratio         → profit_margin = Profit / Sales
    │    ├── Producto      → revenue_total = Sales × Quantity
    │    ├── Diferencia    → costo_implicito = Sales − Profit
    │    ├── Binarias      → es_rentable, tiene_descuento, alto_valor
    │    └── Bins          → rango_descuento (pd.cut), cuartil_ventas (pd.qcut)
    │
    └─ Feature Selection ─────────────────────────────────────────────────────
         ├── SelectKBest(f_regression)  → para predecir Profit en $
         ├── SelectKBest(f_classif)     → para clasificar rentable/no rentable
         └── Dataset final con top K features listo para modelar
```

---

### Herramientas aprendidas

| Herramienta | Librería | Para qué |
|---|---|---|
| `LabelEncoder` | sklearn | Encoding ordinal automático (orden alfabético) |
| `OrdinalEncoder` | sklearn | Encoding ordinal con orden definido manualmente |
| `pd.get_dummies` | pandas | One-Hot Encoding rápido para notebooks |
| `np.where` | numpy | Condicional vectorizado (evita loops y división por cero) |
| `pd.cut` / `pd.qcut` | pandas | Binning por valor definido / por cuantil |
| `SelectKBest(f_regression)` | sklearn | Selección de features para target numérico |
| `SelectKBest(f_classif)` | sklearn | Selección de features para target binario |
| `.get_support()` | sklearn | Extrae la máscara booleana de features seleccionadas |

---

### 🧠 Preguntas para reflexionar

1. Si codificas `Region` con Label Encoding (`East=0, West=1, Central=2, South=3`), ¿qué relación matemática falsa aprende el modelo?
2. ¿Por qué `drop_first=True` en OHE? ¿Qué problema resuelve y cuándo **no** deberías usarlo?
3. `profit_margin` se calcula como `Profit / Sales`. ¿Por qué es más informativa que `Profit` sola al comparar transacciones de montos muy distintos?
4. El F-score de `Discount` fue alto tanto en `f_regression` como en `f_classif`. ¿Eso prueba que el descuento *causa* pérdidas? (Recordemos la clase anterior...)
5. Si aumentas K de 10 a 20 en SelectKBest, ¿el modelo siempre va a mejorar? ¿Por qué sí o por qué no?